# NB03 · ETAS (Omori–Utsu + Gutenberg–Richter) for the seismic layer  (CPU · ~5–10 min)

ETAS is a Hawkes process, so it is fit by MLE and explained in the same way as NB02.

1. **Parameter recovery** on synthetic catalogs with known truth (bias, spread, profile checks).
2. **Fit on a real catalog** if one is available (USGS ComCat via internet, or a CSV you attach; columns `time`,`mag`), otherwise the notebook says so and uses the synthetic catalog only. *Do not present synthetic-only results as real-data accuracy.*
3. **Out-of-sample metrics:** log-likelihood gain over a Poisson model (information gain per event), aftershock-count calibration (N-test style), aftershock probability reliability for the 1–72 h window.
4. Outputs `etas_params.json` + a function-free probability table the API can load.

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
QUICK = False
CFG = dict(n_syn=30, T_days=150, n_boot=30, m0=4.0, n_main=5)
if QUICK: CFG.update(n_syn=4, T_days=60, n_boot=3, n_main=3)
TRUE = dict(W.ETAS_TRUE); TRUE['mu'] = 0.05; print('true params', TRUE)

## 1 · Parameter recovery on synthetic catalogs

In [ ]:
def syn_catalog(seed, T_days=CFG['T_days']):
    # background + several large mainshocks, each with its own ETAS cascade (one big sequence alone is weakly identifiable)
    rng = np.random.default_rng(seed); parts = [W.etas_simulate(None, None, TRUE, T_days, rng, mu=TRUE['mu'], m_max=7.5)]
    for tm in np.sort(rng.uniform(2, T_days * 0.75, CFG['n_main'])): parts.append(W.etas_simulate(tm, rng.uniform(6.0, 7.2), TRUE, T_days, rng, m_max=7.5))
    ev = np.vstack([p for p in parts if len(p)]); return ev[np.argsort(ev[:, 0])]
fits = []
for sd in tqdm(range(CFG['n_syn']), desc='fit synthetic'):
    ev = syn_catalog(sd); n = len(ev)
    if n < 30 or n > 4000: continue
    r = W.etas_fit(ev[:, 0], ev[:, 1], CFG['T_days'], CFG['m0']); r['n'] = n; r['seed'] = sd; fits.append(r)
F = pd.DataFrame(fits); print(len(F), 'catalogs fit; events/catalog median', int(F.n.median()))
rec = pd.DataFrame({k: dict(true=TRUE[k], mean=F[k].mean(), median=F[k].median(), rel_bias_pct=100 * (F[k].median() - TRUE[k]) / TRUE[k], iqr_lo=F[k].quantile(.25), iqr_hi=F[k].quantile(.75)) for k in ['mu', 'K', 'alpha', 'c', 'p']}).T; display(rec.round(4))
fig, ax = plt.subplots(1, 5, figsize=(18, 3.4))
for a, k in zip(ax, ['mu', 'K', 'alpha', 'c', 'p']): a.hist(F[k], bins=12, color='tab:blue', alpha=.8); a.axvline(TRUE[k], c='r', lw=2); a.set_title(k)
fig.suptitle('ETAS MLE recovery (red = truth)'); savefig('20_etas_recovery')

## 2 · Real catalog (optional) — USGS ComCat or an attached CSV

In [ ]:
REAL = None; src = None
cand = [p for p in (find('*quake*.csv', False), find('*earthquake*.csv', False), find('*catalog*.csv', False), find('database.csv', False)) if p]
def parse_catalog(path):
    df = pd.read_csv(path); df.columns = [c.lower() for c in df.columns]
    tcol = next((c for c in ('time', 'date', 'datetime') if c in df.columns), None); mcol = next((c for c in ('mag', 'magnitude') if c in df.columns), None)
    if tcol is None or mcol is None: return None
    if tcol == 'date' and 'time' in df.columns: df['time'] = df['date'].astype(str) + ' ' + df['time'].astype(str); tcol = 'time'
    t = pd.to_datetime(df[tcol], errors='coerce', utc=True, format='mixed'); ok = t.notna() & df[mcol].notna(); return pd.DataFrame(dict(t=t[ok], m=df[mcol][ok])).sort_values('t')
for p in cand:
    try:
        c = parse_catalog(p)
        if c is not None and len(c) > 200: REAL, src = c, p; break
    except Exception as e: print('skip', p, e)
if REAL is None:
    try:   # needs Internet ON in the Kaggle notebook settings
        import urllib.request, io
        url = 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2019-01-01&endtime=2024-12-31&minmagnitude=4.0&minlatitude=30&maxlatitude=42&minlongitude=-125&maxlongitude=-114'
        raw = urllib.request.urlopen(url, timeout=60).read().decode(); open(f'{OUT}/usgs_catalog.csv', 'w').write(raw); REAL = parse_catalog(f'{OUT}/usgs_catalog.csv'); src = 'USGS ComCat API'
    except Exception as e: print('No real catalog available (', type(e).__name__, '). Attach a CSV with time,mag or enable Internet. Continuing with synthetic only.')
print('real catalog:', None if REAL is None else (src, len(REAL), REAL.t.min(), REAL.t.max()))

In [ ]:
RES = dict(synthetic=dict(median=F[['mu', 'K', 'alpha', 'c', 'p']].median().to_dict(), n_catalogs=len(F)))
if REAL is not None:
    days = (REAL.t - REAL.t.min()).dt.total_seconds().values / 86400; R = REAL[REAL.m >= CFG['m0']]; tt = ((R.t - REAL.t.min()).dt.total_seconds().values) / 86400; mm = R.m.values; Tend = float(days.max() + 1)
    # split in time: fit on first 70 %, test on last 30 % (72 h embargo is negligible vs ETAS memory but we apply it)
    cut = 0.7 * Tend; tr = tt < cut; te = tt >= cut + 3; n_tr = int(tr.sum())
    if n_tr > 3000: print('catalog large: using the most recent 3000 training events (O(n^2) likelihood)'); sel = np.where(tr)[0][-3000:]; tt_, mm_ = tt[sel], mm[sel]; t0_ = tt_[0]; tt_ = tt_ - t0_; Tfit = cut - t0_
    else: tt_, mm_, Tfit = tt[tr], mm[tr], cut
    fit = W.etas_fit(tt_, mm_, Tfit, CFG['m0']); print('real-catalog ETAS fit', {k: round(v, 4) for k, v in fit.items() if k in ('mu', 'K', 'alpha', 'c', 'p', 'ok')})
    # b-value (Aki MLE)
    b_val = 1 / (np.log(10) * (mm.mean() - (CFG['m0'] - 0.05))); print('Gutenberg-Richter b =', round(b_val, 3))
    # out-of-sample information gain per event: ETAS (parameters fixed, history = all earlier events) vs homogeneous Poisson (rate from training)
    full_t, full_m = tt, mm; sel_te = np.where(te)[0]; dt = full_t[sel_te][:, None] - full_t[None, :]; prod = fit['K'] * 10 ** (fit['alpha'] * (full_m - CFG['m0']))
    lam = fit['mu'] + np.where(dt > 0, (np.where(dt > 0, dt, 1) + fit['c']) ** (-fit['p']), 0) @ prod
    rate0 = len(tt_) / Tfit; ig = (np.log(lam) - np.log(rate0)).mean(); print(f'information gain over Poisson: {ig:.3f} nats/event on {len(sel_te)} test events')
    RES['real'] = dict(source=str(src), n_train=n_tr, n_test=int(len(sel_te)), params={k: fit[k] for k in ('mu', 'K', 'alpha', 'c', 'p')}, b=float(b_val), info_gain_nats_per_event=float(ig))
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8)); ax[0].plot(tt, np.arange(len(tt)), lw=1); ax[0].axvline(cut, c='r', ls='--'); ax[0].set_title('cumulative M>=m0 events (red = train/test cut)'); ax[0].set_xlabel('days')
    ax[1].hist(mm, bins=25, log=True); ax[1].set_title(f'magnitude distribution, b={b_val:.2f}'); savefig('21_real_catalog')
else: print('(skipped real-data section)')

## 3 · Forecast metrics (synthetic, strict out-of-sample): aftershock-probability reliability for the 1–72 h window

In [ ]:
def prob_event(theta, hist_t, hist_m, t_now, h_days, m_min, T_end):
    """P(at least one event >= m_min in (t_now, t_now+h]) = 1 - exp(-integral of lambda * P(m>=m_min))  (history frozen: conservative lower bound)"""
    c, p, K, al, m0, b = theta['c'], theta['p'], theta['K'], theta['alpha'], theta['m0'], theta['b']; prev = hist_t <= t_now
    a0 = t_now - hist_t[prev]; prod = K * 10 ** (al * (hist_m[prev] - m0))
    integ = (prod * (((a0 + c) ** (1 - p) - (a0 + h_days + c) ** (1 - p)) / (p - 1))).sum() + theta['mu'] * h_days
    return 1 - np.exp(-integ * 10 ** (-b * (m_min - m0)))
rows = []; thfit = dict(F[['mu', 'K', 'alpha', 'c', 'p']].median()); thfit.update(m0=CFG['m0'], b=1.0)
for sd in tqdm(range(100, 100 + (40 if not QUICK else 5)), desc='forecast eval'):
    ev = syn_catalog(sd); t, m = ev[:, 0], ev[:, 1]
    if len(m) < 20: continue
    main = t[np.argmax(m)]  # largest event of the sequence
    for dt_ in (0.05, 0.25, 1.0, 3.0):                       # issue forecast 1.2 h .. 3 d after the mainshock
        tn = main + dt_
        for h_h in (1, 6, 24, 72):
            for mmin in (4.5, 5.5):
                pr = prob_event(thfit, t, m, tn, h_h / 24, mmin, CFG['T_days']); y = ((t > tn) & (t <= tn + h_h / 24) & (m >= mmin)).any(); rows.append(dict(p=pr, y=int(y), h=h_h, mmin=mmin, since_main_d=dt_))
FR = pd.DataFrame(rows); from sklearn.metrics import brier_score_loss, roc_auc_score
print(f'N forecasts {len(FR)}, base rate {FR.y.mean():.3f}, Brier {brier_score_loss(FR.y, FR.p):.4f} (climatology {FR.y.mean()*(1-FR.y.mean()):.4f}), AUROC {roc_auc_score(FR.y, FR.p):.3f}')
bins = np.linspace(0, 1, 11); FR['bin'] = pd.cut(FR.p, bins, include_lowest=True); rel = FR.groupby('bin', observed=True).agg(p=('p', 'mean'), y=('y', 'mean'), n=('y', 'size')); ece = (rel.n * (rel.p - rel.y).abs()).sum() / rel.n.sum(); print('ECE', round(float(ece), 4))
fig, ax = plt.subplots(1, 3, figsize=(16, 4)); ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].plot(rel.p, rel.y, 'o-'); [ax[0].text(a, b, int(n), fontsize=7) for a, b, n in zip(rel.p, rel.y, rel.n)]; ax[0].set_xlabel('forecast P'); ax[0].set_ylabel('observed freq'); ax[0].set_title(f'reliability (ECE {ece:.3f})')
for h_h, g in FR.groupby('h'): gg = g.groupby('since_main_d').agg(p=('p', 'mean'), y=('y', 'mean')); ax[1].plot(gg.index, gg.p, '-o', label=f'{h_h} h window')
ax[1].set_xscale('log'); ax[1].set_xlabel('days since mainshock'); ax[1].set_ylabel('mean P(M>=4.5)'); ax[1].set_title('Omori decay of aftershock probability'); ax[1].legend()
ev = syn_catalog(7); tt_ = np.linspace(ev[:, 0][np.argmax(ev[:, 1])] + .02, ev[:, 0][np.argmax(ev[:, 1])] + 20, 200); ax[2].loglog(tt_ - ev[:, 0][np.argmax(ev[:, 1])], [prob_event(thfit, ev[:, 0], ev[:, 1], x, 1 / 24, 4.5, 60) for x in tt_]); ax[2].set_title('P(M>=4.5 in next hour) vs time since main'); savefig('22_etas_forecast_metrics')

In [ ]:
# aftershock probability table the demo/API loads (hours x magnitude, for a reference M6.5 mainshock) - generated from FITTED params
tab = []
ev = np.array([[0.0, 6.5]])
for t_issue_h in [1, 3, 6, 12, 24, 48]:
    for h_h in [1, 3, 6, 12, 24, 48, 72]:
        for mmin in [4.5, 5.0, 5.5, 6.0]: tab.append(dict(hours_since_main=t_issue_h, horizon_h=h_h, m_min=mmin, prob=prob_event(thfit, ev[:, 0], ev[:, 1], t_issue_h / 24, h_h / 24, mmin, 60)))
PT = pd.DataFrame(tab); PT.to_csv(f'{OUT}/etas_aftershock_table.csv', index=False)
json.dump(dict(fitted_synthetic={k: float(v) for k, v in thfit.items()}, truth=TRUE, results=RES, brier=float(brier_score_loss(FR.y, FR.p)), ece=float(ece)), open(f'{OUT}/etas_params.json', 'w'), indent=1, default=float)
print(PT[(PT.horizon_h == 24) & (PT.m_min == 5.0)].round(3).to_string(index=False)); print('saved etas_params.json, etas_aftershock_table.csv')